# Autoregressive decoding and KV-cache anatomy

Implements the closed-form byte-count for a transformer KV cache, runs
greedy generation with and without the cache on SmolLM2-135M, and verifies
the measured peak memory tracks the formula while cached decoding delivers
the expected wall-clock speedup.

**Learning objectives**
- Derive and verify ``kv_cache_bytes = 2·L·H·D·T·B·bytes``.
- Generate with and without a `DynamicCache`, compare token-for-token.
- Sweep context length 128 → 2048 and read off peak memory and tokens/sec.
- Classify decode as memory-bound using the roofline from the GPU track.

**Papers**: 2309.06180 (PagedAttention).
**Hardware**: CPU or T4. On CPU the per-token latency is ~10× higher; the
scoring thresholds are hardware-invariant (ratio-based).
**Estimated runtime**: ≈20 min on T4, ≈6 min on CPU for the shortest settings.


In [ ]:
from __future__ import annotations

import math
import os
import sys
import time
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import torch

from llm_infra_lab._utils import get_device, hardware_check, set_seed
from llm_infra_lab.models import get as get_model
from scoring.harness import Scorer

set_seed(0)
s = Scorer("01_inference_01_autoregressive_decoding_kv_cache")
print(hardware_check())
DEVICE = get_device()


In [ ]:
def kv_cache_bytes(
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    seq_len: int,
    batch: int = 1,
    dtype_bytes: int = 2,
) -> int:
    '''Closed-form KV-cache size in bytes.

    2 comes from storing both K and V, ``dtype_bytes`` is the per-element
    storage (2 for fp16/bf16, 1 for int8).
    '''
    return 2 * num_layers * num_kv_heads * head_dim * seq_len * batch * dtype_bytes


# Unit checks against a known reference (Llama-3.1-8B fp16, B=1, T=4096):
# 32 layers, 8 KV heads (GQA), head_dim 128 -> 2*32*8*128*4096*2 = 512 MB.
ref = kv_cache_bytes(32, 8, 128, 4096)
print(f"reference 8B KV @ 4k = {ref / 1024**2:.0f} MiB")
s.check("kv_formula_matches_hand_calc", lambda: ref == 2 * 32 * 8 * 128 * 4096 * 2)
s.check("kv_formula_linear_in_seq", lambda: kv_cache_bytes(32, 8, 128, 8192) == 2 * ref)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

spec = get_model("smollm2-135m")
print(f"loading {spec.hf_id}")
tok = AutoTokenizer.from_pretrained(spec.hf_id)
model = AutoModelForCausalLM.from_pretrained(spec.hf_id, torch_dtype=torch.float32).to(DEVICE)
model.eval()

cfg = model.config
NUM_LAYERS = cfg.num_hidden_layers
NUM_KV_HEADS = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
HEAD_DIM = cfg.hidden_size // cfg.num_attention_heads
DTYPE_BYTES = 2 if model.dtype in (torch.float16, torch.bfloat16) else 4
print(f"L={NUM_LAYERS} H_kv={NUM_KV_HEADS} D={HEAD_DIM} dtype_bytes={DTYPE_BYTES}")


In [ ]:
from transformers import DynamicCache


@torch.inference_mode()
def generate_no_cache(prompt_ids: torch.Tensor, n_new: int) -> tuple[torch.Tensor, float]:
    '''Greedy decode without a KV cache: re-feeds the full context each step.'''
    ids = prompt_ids.clone()
    t0 = time.perf_counter()
    for _ in range(n_new):
        out = model(ids)
        next_id = out.logits[:, -1].argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=-1)
    return ids, time.perf_counter() - t0


@torch.inference_mode()
def generate_with_cache(prompt_ids: torch.Tensor, n_new: int) -> tuple[torch.Tensor, float, DynamicCache]:
    '''Greedy decode reusing a `DynamicCache` — the prefill feeds the full prompt once.'''
    cache = DynamicCache()
    ids = prompt_ids.clone()
    t0 = time.perf_counter()
    out = model(ids, past_key_values=cache, use_cache=True)
    cache = out.past_key_values
    next_id = out.logits[:, -1].argmax(-1, keepdim=True)
    ids = torch.cat([ids, next_id], dim=-1)
    for _ in range(n_new - 1):
        out = model(next_id, past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        next_id = out.logits[:, -1].argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=-1)
    return ids, time.perf_counter() - t0, cache


prompt = "The key idea behind the Transformer architecture is"
prompt_ids = tok(prompt, return_tensors="pt").input_ids.to(DEVICE)
N_NEW = 32

out_no, dt_no = generate_no_cache(prompt_ids, N_NEW)
out_yes, dt_yes, final_cache = generate_with_cache(prompt_ids, N_NEW)
print(f"no-cache:   {dt_no * 1000:.1f} ms ({N_NEW / dt_no:.1f} tok/s)")
print(f"with-cache: {dt_yes * 1000:.1f} ms ({N_NEW / dt_yes:.1f} tok/s)")
print("text:", tok.decode(out_yes[0], skip_special_tokens=True))

s.check(
    "greedy_outputs_agree",
    lambda: torch.equal(out_no, out_yes),
    msg="no-cache and cached greedy must produce identical token ids",
)
s.benchmark(
    "cache_speedup_at_least_3x",
    measured=N_NEW / dt_yes,
    baseline=N_NEW / dt_no,
    must_beat=3.0,
    unit="tok/s",
    higher_is_better=True,
)


In [ ]:
CONTEXTS = [128, 256, 512]
if IS_CUDA := torch.cuda.is_available():
    # T4 can handle the longer contexts; CPU stops at 512 to keep the notebook under budget.
    CONTEXTS = [128, 256, 512, 1024, 2048]

filler_prompt = tok("Transformers " * 4096, return_tensors="pt").input_ids[0].to(DEVICE)

mem_measurements: dict[int, dict[str, float]] = {}
tok_per_s: dict[int, float] = {}

for ctx in CONTEXTS:
    ids = filler_prompt[:ctx].unsqueeze(0)
    predicted_mb = kv_cache_bytes(
        NUM_LAYERS, NUM_KV_HEADS, HEAD_DIM, seq_len=ctx + 16, batch=1, dtype_bytes=4
    ) / 1024**2
    if IS_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    out, dt, _ = generate_with_cache(ids, n_new=16)
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if IS_CUDA else float("nan")
    mem_measurements[ctx] = {"predicted_mb": predicted_mb, "peak_mb": peak_mb}
    tok_per_s[ctx] = 16 / dt
    print(f"ctx={ctx:>4}  predicted={predicted_mb:6.1f} MiB  peak={peak_mb:6.1f} MiB  {tok_per_s[ctx]:5.1f} tok/s")

# On CPU the peak-allocated metric is unavailable — skip the memory check there.
if IS_CUDA:
    # The peak includes weights + activations + KV; KV should at least be present
    # and scale roughly linearly with context.
    peak_mem_vals = [mem_measurements[c]["peak_mb"] for c in CONTEXTS]
    delta_mem = peak_mem_vals[-1] - peak_mem_vals[0]
    delta_ctx = CONTEXTS[-1] - CONTEXTS[0]
    expected_delta = (
        kv_cache_bytes(NUM_LAYERS, NUM_KV_HEADS, HEAD_DIM, delta_ctx + 16, batch=1, dtype_bytes=4)
        - kv_cache_bytes(NUM_LAYERS, NUM_KV_HEADS, HEAD_DIM, 16, batch=1, dtype_bytes=4)
    ) / 1024**2
    print(f"Δpeak={delta_mem:.1f} MiB expected Δ={expected_delta:.1f} MiB")
    s.assert_close(
        "peak_memory_growth_matches_formula",
        actual=delta_mem,
        expected=expected_delta,
        rtol=0.35,
    )
else:
    s.skip("peak_memory_growth_matches_formula", "peak-memory metric requires CUDA")

# Tokens/sec should stay finite (decode is memory-bound so drops with context, but gently).
s.check(
    "tok_per_s_nondegenerate_all_contexts",
    lambda: all(tok_per_s[c] > 0.5 for c in CONTEXTS),
    msg=f"measured {tok_per_s}",
)


In [ ]:
# Decode-step arithmetic intensity: a single-token forward reads the weights
# from HBM and a KV tile; at batch=1 the intensity is O(1) → memory-bound.
# Prefill over a long context tiles large matmuls and is compute-bound.
#
# For Llama-like models, weights_bytes ≈ 2 · P · dtype_bytes (reading them
# once per forward). At batch 1, FLOPs/byte ≈ 1 for decode, ≈100+ for prefill.
NUM_PARAMS = sum(p.numel() for p in model.parameters())
DECODE_AI = 2 / DTYPE_BYTES  # ~1 FLOP per weight byte -> ~1
PREFILL_AI = 2 * 512 / DTYPE_BYTES  # rough for T=512 prefill on this model

print(f"params={NUM_PARAMS/1e6:.1f}M decode AI≈{DECODE_AI:.1f} prefill AI≈{PREFILL_AI:.0f}")
s.check(
    "decode_is_memory_bound",
    lambda: DECODE_AI < 50,
    msg=f"decode AI={DECODE_AI:.1f} should be well below typical GPU ridge (~50-100)",
)
s.check(
    "prefill_is_compute_bound",
    lambda: PREFILL_AI > 100,
    msg=f"prefill AI={PREFILL_AI:.0f} should be above typical GPU ridge",
)


In [ ]:
s.summary()
s.save()
